# Basic Statistical Analysis

This notebook performs basic statistical analysis on Monte Carlo simulation results.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
from scipy import stats

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Load data
stats_dir = project_root / "stats"
analysis_files = list(stats_dir.glob("analysis_*.json")) if stats_dir.exists() else []

if analysis_files:
    latest_file = max(analysis_files, key=lambda p: p.stat().st_mtime)
    with open(latest_file, 'r') as f:
        data = json.load(f)
    print(f"Loaded: {latest_file.name}")
else:
    print("No data files found")
    data = None

## Summary Statistics

In [ ]:
if data:
    summary = data.get('summary', {})
    
    print("=" * 60)
    print("SUMMARY STATISTICS")
    print("=" * 60)
    
    print(f"\nTotal Games: {summary.get('total_games', 'N/A')}")
    print(f"Team 0 Wins: {summary.get('team0_wins', 'N/A')}")
    print(f"Team 1 Wins: {summary.get('team1_wins', 'N/A')}")
    
    # Hands played statistics
    hands_stats = summary.get('hands_played', {})
    if hands_stats:
        print(f"\nHands Played:")
        print(f"  Mean: {hands_stats.get('mean', 'N/A'):.2f}")
        print(f"  Std: {hands_stats.get('std', 'N/A'):.2f}")
        print(f"  Min: {hands_stats.get('min', 'N/A')}")
        print(f"  Max: {hands_stats.get('max', 'N/A')}")
        print(f"  Median: {hands_stats.get('median', 'N/A'):.2f}")
    
    # Score statistics
    for team in [0, 1]:
        score_key = f'scores_team{team}'
        score_stats = summary.get(score_key, {})
        if score_stats:
            print(f"\nTeam {team} Scores:")
            print(f"  Mean: {score_stats.get('mean', 'N/A'):.2f}")
            print(f"  Std: {score_stats.get('std', 'N/A'):.2f}")
            print(f"  Min: {score_stats.get('min', 'N/A')}")
            print(f"  Max: {score_stats.get('max', 'N/A')}")

## Win Rate Analysis

In [ ]:
if data:
    win_rates = data.get('win_rates_by_combination', {})
    
    if win_rates:
        print("=" * 60)
        print("WIN RATES BY RISK COMBINATION")
        print("=" * 60)
        
        # Convert to DataFrame for easier analysis
        win_rate_data = []
        for risk_key, rates in win_rates.items():
            win_rate_data.append({
                'risk_combination': str(risk_key),
                'team0_rate': rates.get('team0_rate', 0),
                'team1_rate': rates.get('team1_rate', 0),
                'total_games': rates.get('total_games', 0),
            })
        
        df_win_rates = pd.DataFrame(win_rate_data)
        df_win_rates = df_win_rates.sort_values('total_games', ascending=False)
        
        print("\nTop 10 Most Tested Combinations:")
        print(df_win_rates.head(10).to_string(index=False))
        
        print("\n\nBest Performing Combinations (Team 0):")
        print(df_win_rates.nlargest(10, 'team0_rate')[['risk_combination', 'team0_rate', 'total_games']].to_string(index=False))
        
        print("\n\nBest Performing Combinations (Team 1):")
        print(df_win_rates.nlargest(10, 'team1_rate')[['risk_combination', 'team1_rate', 'total_games']].to_string(index=False))